# CareerLens AI

# Notebook 04 — Model Optimization

## Objective

Optimize the baseline Random Forest model using hyperparameter tuning.

In this notebook we will:

- Load processed train/test data
- Load the baseline model results
- Tune Random Forest using RandomizedSearchCV
- Evaluate the optimized model
- Compare baseline vs tuned performance
- Save the optimized model

In [1]:
import joblib
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
X_train = joblib.load("../datasets/processed/X_train.pkl")
X_test = joblib.load("../datasets/processed/X_test.pkl")

y_train = joblib.load("../datasets/processed/y_train.pkl")
y_test = joblib.load("../datasets/processed/y_test.pkl")

label_encoder = joblib.load("../models/label_encoder.pkl")
baseline_model = joblib.load("../models/best_model.pkl")

print("Files loaded successfully.")
print(X_train.shape, X_test.shape)

Files loaded successfully.
(1987, 5000) (497, 5000)


In [3]:
baseline_predictions = baseline_model.predict(X_test)

baseline_metrics = {
    "Model": "Baseline Random Forest",
    "Accuracy": accuracy_score(y_test, baseline_predictions),
    "Precision": precision_score(y_test, baseline_predictions, average="weighted", zero_division=0),
    "Recall": recall_score(y_test, baseline_predictions, average="weighted", zero_division=0),
    "F1 Score": f1_score(y_test, baseline_predictions, average="weighted", zero_division=0)
}

baseline_metrics

{'Model': 'Baseline Random Forest',
 'Accuracy': 0.7706237424547284,
 'Precision': 0.7819245997920901,
 'Recall': 0.7706237424547284,
 'F1 Score': 0.7529893003187409}

In [4]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 10, 20, 30, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [7]:
random_forest = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    estimator=random_forest,
    param_distributions=param_distributions,
    n_iter=10,
    scoring="f1_weighted",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=1
)

random_search.fit(X_train, y_train)

print("Hyperparameter tuning completed.")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=100; total time=   1.5s
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=100; total time=   1.5s
[CV] END max_depth=30, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=100; total time=   1.4s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=300; total time=   2.0s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=300; total time=   2.1s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=1, min_samples_split=5, n_estimators=300; total time=   2.0s
[CV] END max_depth=30, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   0.7s
[CV] END max_depth=30, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_est

In [8]:
random_search.best_params_

{'n_estimators': 500,
 'min_samples_split': 2,
 'min_samples_leaf': 2,
 'max_features': 'sqrt',
 'max_depth': None}

In [9]:
tuned_model = random_search.best_estimator_

tuned_predictions = tuned_model.predict(X_test)

tuned_metrics = {
    "Model": "Tuned Random Forest",
    "Accuracy": accuracy_score(y_test, tuned_predictions),
    "Precision": precision_score(y_test, tuned_predictions, average="weighted", zero_division=0),
    "Recall": recall_score(y_test, tuned_predictions, average="weighted", zero_division=0),
    "F1 Score": f1_score(y_test, tuned_predictions, average="weighted", zero_division=0)
}

tuned_metrics

{'Model': 'Tuned Random Forest',
 'Accuracy': 0.7746478873239436,
 'Precision': 0.7903557854055404,
 'Recall': 0.7746478873239436,
 'F1 Score': 0.7563037144269039}

In [10]:
comparison_df = pd.DataFrame([
    baseline_metrics,
    tuned_metrics
])

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Baseline Random Forest,0.770624,0.781925,0.770624,0.752989
1,Tuned Random Forest,0.774648,0.790356,0.774648,0.756304


In [11]:
joblib.dump(
    tuned_model,
    "../models/tuned_random_forest.pkl"
)

print("Tuned model saved.")

Tuned model saved.


In [12]:
comparison_df.to_csv(
    "../reports/tuning_results.csv",
    index=False
)

print("Results saved.")

Results saved.


# Day 6 Final Observations

## What we completed

- Loaded the baseline Random Forest model.
- Created a hyperparameter search space.
- Used RandomizedSearchCV for model optimization.
- Evaluated the tuned model.
- Compared baseline and tuned model performance.
- Saved the optimized model.
- Saved optimization results.

## Engineering Decision

If the tuned model performs better than the baseline, it will be used for the next stage.

If the tuned model does not improve performance significantly, the baseline Random Forest may remain the selected model.

## Next Step

The next phase will focus on error analysis and explainability.